In [3]:
import numpy as np
import kagglehub
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# --- STEP 1: Download Data ---
path = kagglehub.dataset_download("samasiayushman/doraemon-complete-data")
print("Dataset downloaded to:", path)

# --- STEP 2: Preprocessing (Conceptual) ---
# ในขั้นตอนนี้คุณต้องโหลดภาพจาก path แล้วแปลงเป็น Matrix
# สมมติว่าเรามีข้อมูล X (ภาพ 64x64 Flatten เป็น 4096) และ y (Label) เรียบร้อยแล้ว
# X = np.random.rand(1000, 4096) 
# y = np.random.randint(0, 2, (1000, 1))

# --- STEP 3 & 4: Model & Training Engine ---
class DoraemonNet:
    def __init__(self, input_size, hidden_size, output_size):
        # He Initialization สำหรับ Weight
        self.W1 = np.random.randn(input_size, hidden_size) * np.sqrt(2./input_size)
        self.W2 = np.random.randn(hidden_size, output_size) * np.sqrt(2./hidden_size)
        self.loss_history = []

    def relu(self, x):
        return np.maximum(0, x)

    def relu_derivative(self, x):
        return (x > 0).astype(float)

    def sigmoid(self, x):
        return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

    def forward(self, X):
        self.z1 = np.dot(X, self.W1)
        self.a1 = self.relu(self.z1)  # Hidden Layer Activation
        self.z2 = np.dot(self.a1, self.W2)
        self.a2 = self.sigmoid(self.z2) # Output Layer Activation
        return self.a2

    def backward(self, X, y, output, lr):
        m = y.shape[0]
        
        # Error ที่ Output
        dz2 = output - y 
        dW2 = (1/m) * np.dot(self.a1.T, dz2)
        
        # Error ที่ Hidden Layer (Backpropagate)
        dz1 = np.dot(dz2, self.W2.T) * self.relu_derivative(self.a1)
        dW1 = (1/m) * np.dot(X.T, dz1)
        
        # Update Weights
        self.W2 -= lr * dW2
        self.W1 -= lr * dW1

    def train(self, X, y, epochs, lr):
        for i in range(epochs):
            output = self.forward(X)
            self.backward(X, y, output, lr)
            
            # คำนวณ Binary Cross Entropy Loss
            loss = -np.mean(y * np.log(output + 1e-8) + (1-y) * np.log(1-output + 1e-8))
            self.loss_history.append(loss)
            
            if i % 100 == 0:
                print(f"Epoch {i}, Loss: {loss:.4f}")

# --- STEP 5: Evaluation ---
def evaluate_model(model, X_test, y_test):
    predictions = (model.forward(X_test) > 0.5).astype(int)
    
    print("\n--- Classification Report ---")
    print(classification_report(y_test, predictions))
    
    # Plotting Loss
    plt.plot(model.loss_history)
    plt.title("Training Loss over Time")
    plt.xlabel("Epochs")
    plt.ylabel("Loss")
    plt.show()

# สั่งรัน (ตัวอย่าง)
# model = DoraemonNet(4096, 64, 1)
# model.train(X_train, y_train, epochs=1000, lr=0.01)
# evaluate_model(model, X_test, y_test)

Dataset downloaded to: C:\Users\USER\.cache\kagglehub\datasets\samasiayushman\doraemon-complete-data\versions\3
